# NIR White Lagkitan Corn (Binary-Class)

In [ ]:
import pandas as pd
import numpy as np

## Load Data

In [ ]:
df_raw_data = pd.read_csv('appendix_raw_dataset2.csv')
df_raw_data

## Preprocessing

In [ ]:
feature_cols = ['730nm','760nm','810nm','860nm','900nm','940nm']

### Outlier Removal

In [ ]:
def filter_scans_stage1(df, keep_ratio_fallback=0.85):
    filtered_parts = []

    for sample_id, group in df.groupby('Sample No.'):
        X = group[feature_cols].values

        centroid = X.mean(axis=0)

        dists = np.linalg.norm(X - centroid, axis=1)

        med = np.median(dists)
        mad = np.median(np.abs(dists - med))

        threshold = med + 1.5 * mad

        mask = dists <= threshold

        if mask.sum() < len(group) * 0.5:
            cutoff = np.quantile(dists, keep_ratio_fallback)
            mask = dists <= cutoff

        filtered_group = group[mask]
        filtered_parts.append(filtered_group)

    return pd.concat(filtered_parts, ignore_index=True)

df_stage1 = filter_scans_stage1(df_raw_data)

### SNV

In [ ]:
df_snv = df_stage1.copy()

def snv_row(x):
    x = x.astype(float)
    mean = x.mean()
    std = x.std(ddof=1)

    if std < 1e-8:
        return x * np.nan

    return (x - mean) / std

df_snv[feature_cols] = df_snv[feature_cols].apply(
    snv_row, axis=1, result_type='expand'
)

df_snv = df_snv.dropna()

### Median Aggregation

In [ ]:
def stage2_median_aggregation(df):

    df_agg = df.groupby('Sample No.')[feature_cols].median().reset_index()

    class_map = df.groupby('Sample No.')['Class'].first().reset_index()

    df_final = pd.merge(df_agg, class_map, on='Sample No.')

    return df_final

df_stage2 = stage2_median_aggregation(df_snv)

## Data Preparation

### Selecting Features (X) and Targets (y)

In [ ]:
X = df_stage2[feature_cols]
y = df_stage2['Class']

### Encode Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
dict(zip(le.classes_, le.transform(le.classes_)))

### Data Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42)
X_val, X_test, y_val , y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

## Optuna Hyperparameter Tuning

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier

# For reproducibility
sampler = optuna.samplers.TPESampler(seed=42)
inner_cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

# Tune to penalize instability
alpha = 0.5
stability_std_cap = 0.02

### Logistic Regression

In [ ]:
def objective_lr(trial):
    params = {
        'C': trial.suggest_float('C', 0.05, 3.0, log=True),

        'penalty': 'l2',

        'solver': trial.suggest_categorical(
            'solver', ['liblinear', 'lbfgs']
        ),

        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', None]
        ),

        'max_iter': 5000,
        'tol': trial.suggest_float('tol', 1e-5, 1e-3, log=True),
        'random_state': 42
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [ ]:
study_lr = optuna.create_study(direction='maximize', sampler=sampler)
study_lr.optimize(objective_lr, n_trials=100)

study_lr.best_params, study_lr.best_value

### Random Forest

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 350),

        'max_depth': trial.suggest_int('max_depth', 2, 5),

        'max_features': trial.suggest_categorical(
            'max_features', ['sqrt', 0.5]
        ),

        'min_samples_split': trial.suggest_int('min_samples_split', 5, 15),

        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),

        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 8, 25),

        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', None]
        ),

        'bootstrap': True,
        'n_jobs': -1,
        'random_state': 42
    }

    model = Pipeline([
        ('clf', RandomForestClassifier(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [ ]:
study_rf = optuna.create_study(direction='maximize', sampler=sampler)
study_rf.optimize(objective_rf, n_trials=100)

study_rf.best_params, study_rf.best_value

### Linear Discriminant Analysis

In [ ]:
def objective_lda(trial):
    solver = trial.suggest_categorical('solver', ['lsqr', 'eigen'])

    params = {
        'solver': solver,

        'shrinkage': trial.suggest_float(
            'shrinkage',
            0.05, 0.8,
            log=True
        ),

        'store_covariance': False
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [ ]:
study_lda = optuna.create_study(direction='maximize', sampler=sampler)
study_lda.optimize(objective_lda, n_trials=100)

study_lda.best_params, study_lda.best_value

### Support Vector Machine

In [ ]:
def objective_svc(trial):
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])

    C = trial.suggest_float('C', 0.05, 5.0, log=True)

    params = {
        'C': C,
        'kernel': kernel,
        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', None]
        ),
    }

    if kernel == 'rbf':
        params['gamma'] = trial.suggest_float(
            'gamma',
            1e-4,
            0.01,
            log=True
        )

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [ ]:
study_svc = optuna.create_study(direction='maximize', sampler=sampler)
study_svc.optimize(objective_svc, n_trials=100)

study_svc.best_params, study_svc.best_value

### XGBoost

In [ ]:
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 3),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.02,
            0.05,
            log=True
        ),

        'n_estimators': trial.suggest_int('n_estimators', 80, 180),

        'min_child_weight': trial.suggest_float('min_child_weight', 3.0, 10.0),

        'gamma': trial.suggest_float('gamma', 0.1, 0.3),

        'subsample': trial.suggest_float('subsample', 0.8, 1.0),

        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),

        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),

        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 0.1),

        'max_delta_step': trial.suggest_int('max_delta_step', 0, 2),

        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss'
    }

    model = Pipeline([
        ('clf', XGBClassifier(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc')

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [ ]:
study_xgb = optuna.create_study(direction='maximize', sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=100)

study_xgb.best_params, study_xgb.best_value

## Validation Set Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**study_lr.best_params))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(**study_rf.best_params))
    ]),
    'Linear Discriminant Analysis': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis(**study_lda.best_params))
    ]),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(**study_svc.best_params))
    ]),
    'XGBoost': Pipeline([
        ('clf', XGBClassifier(**study_xgb.best_params))
    ])
}

In [ ]:
val_predictions = {}
val_metrics = {}
val_reports ={}

for model_name, model in models.items():

    model.fit(X_train, y_train)

    val_predictions[model_name] = model.predict(X_val)

    val_metrics[model_name] = {
        'Accuracy': accuracy_score(y_val, val_predictions[model_name]),
        'Precision': precision_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
        'Recall': recall_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
        'F1': f1_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
    }

    val_reports[model_name] = classification_report(y_val, val_predictions[model_name], zero_division=0)

### Summary Metrics

In [ ]:
df_val_metrics = pd.DataFrame(val_metrics)
df_val_metrics.T.sort_values(by='Accuracy', ascending=False)

### Classification Report

In [ ]:
for model_names, reports in val_reports.items():
    print(f'{model_names}:\n {reports} \n -------------------------------------------------------')

## Test Set Metrics

In [ ]:
X_combined = np.concatenate([X_train, X_val], axis=0)
y_combined = np.concatenate([y_train, y_val], axis=0)

In [ ]:
final_model = XGBClassifier(**study_xgb.best_params)
final_model.fit(X_combined, y_combined)

### Summary Metrics

In [ ]:
test_predictions = final_model.predict(X_test)

test_metrics = {
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, test_predictions),
    'Precision': precision_score(y_test, test_predictions, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, test_predictions, average='weighted', zero_division=0),
    'F1': f1_score(y_test, test_predictions, average='weighted', zero_division=0),
}

In [ ]:
df_test_metrics = pd.DataFrame([test_metrics])
df_test_metrics

### Classification Report

In [ ]:
print(classification_report(y_test, test_predictions))